# 15 — RV-PBS embeddings (RQ4)

Embeds the RV-PBS crops with the same encoder and preprocessing as notebook 03, for the cross-source shift analysis (`scripts/16_external_analysis.py`).

**Input:** `external_crops.zip` (from `scripts/14_crop_rvpbs.py`) in `MyDrive/certified-adaptive-pbs/`.
**Runtime:** T4 GPU, about 2 min. **Output:** `external_v1/` → `results/external_v1/`.


In [ ]:
!pip -q install timm==1.0.* 2>/dev/null
import os, json, time, hashlib, platform, zipfile
import numpy as np, pandas as pd, torch, timm
from google.colab import drive; drive.mount('/content/drive')
D = '/content/drive/MyDrive/certified-adaptive-pbs'; OUT = f'{D}/external_v1'; os.makedirs(OUT, exist_ok=True)
assert os.path.exists(f'{D}/external_crops.zip'), 'upload data/external_crops.zip to MyDrive/certified-adaptive-pbs/'
zipfile.ZipFile(f'{D}/external_crops.zip').extractall('/content/ext')
h = hashlib.sha256(open(f'{D}/external_crops.zip','rb').read()).hexdigest()
meta = pd.read_csv('/content/ext/rv_pbs_crops.csv')
meta['path'] = '/content/ext/rv_pbs_crops/' + meta.crop_id + '.jpg'
assert meta.path.map(os.path.exists).all()
print(len(meta), 'crops |', meta.is_blast.sum(), 'blasts | GPU:', torch.cuda.is_available())

In [ ]:
# identical to 03_embeddings_colab.ipynb cells 4-5
SEED = 20260919; torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
MODEL_NAME = 'vit_small_patch14_dinov2.lvd142m'; IMG_SIZE = 224
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0, img_size=IMG_SIZE).eval().to(device)
cfg = timm.data.resolve_data_config({}, model=model); MEAN, STD = cfg['mean'], cfg['std']
from PIL import Image
import torchvision.transforms.functional as TF
from scipy import ndimage
def quality_features(rgb):
    x = rgb.astype(np.float32) / 255.0; gray = x @ np.array([0.299, 0.587, 0.114], np.float32)
    lap = ndimage.laplace(gray); hsv = np.array(Image.fromarray(rgb).convert('HSV'), np.float32) / 255.0
    return dict(q_brightness=float(gray.mean()), q_contrast=float(gray.std()), q_blur_lapvar=float(lap.var()),
                q_r=float(x[...,0].mean()), q_g=float(x[...,1].mean()), q_b=float(x[...,2].mean()),
                q_sat=float(hsv[...,1].mean()), q_hue=float(hsv[...,0].mean()))
E, Q = [], []
with torch.inference_mode():
    for i in range(0, len(meta), 64):
        ims = [Image.open(p).convert('RGB') for p in meta.path[i:i+64]]
        Q += [quality_features(np.asarray(im)) for im in ims]
        x = torch.stack([TF.normalize(TF.to_tensor(TF.resize(im, [IMG_SIZE, IMG_SIZE], antialias=True)), MEAN, STD) for im in ims]).to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
            E.append(model(x).float().cpu().numpy())
E = np.concatenate(E); assert E.shape[0] == len(meta) and np.isfinite(E).all(); print(E.shape)

In [ ]:
np.save(f'{OUT}/rvpbs_embeddings_dinov2_vits14.npy', E.astype(np.float16))
out = meta.drop(columns=['path']).copy(); out[list(Q[0].keys())] = pd.DataFrame(Q).values
out.to_csv(f'{OUT}/rvpbs_features.csv', index=False)
json.dump(dict(notebook='15_external_embeddings_colab.ipynb', model=MODEL_NAME, img_size=IMG_SIZE, mean=list(MEAN), std=list(STD),
               zip_sha256=h, n=int(len(meta)), torch=torch.__version__, timm=timm.__version__, python=platform.python_version(),
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
               note='same encoder and preprocessing as KU-Optofil embeddings_v1; classifier is applied locally'),
          open(f'{OUT}/config.json', 'w'), indent=1)
print('saved to', OUT)